In [0]:
%pip install databricks-feature-engineering
dbutils.library.restartPython()

In [0]:
# from pyspark.sql.functions import col, monotonically_increasing_id
# from databricks.feature_engineering import FeatureEngineeringClient


# class FeatureTablePreparer:

#     def __init__(self, spark, source_table, target_table, key_columns, surrogate_key_column):
#         self.spark = spark
#         self.source_table = source_table
#         self.target_table = target_table
#         self.key_columns = key_columns
#         self.surrogate_key_column = surrogate_key_column
#         self.df = None

#     def read_source(self):
#         self.df = self.spark.table(self.source_table)
#         self.df = self.df.withColumn("transaction_date", col("transaction_date").cast("date"))

#     def validate_key_uniqueness(self):
#         total_rows = self.df.count()
#         distinct_key_rows = self.df.select(*self.key_columns).distinct().count()

#         if total_rows != distinct_key_rows:
#             raise ValueError(
#                 "Natural key columns " + str(self.key_columns) +
#                 " are NOT unique. Total rows: " + str(total_rows) +
#                 ", distinct key combinations: " + str(distinct_key_rows) +
#                 ". Cannot proceed."
#             )

#         print("Natural key uniqueness validated: " + str(total_rows) + " rows, all unique on " + str(self.key_columns))

#     def add_surrogate_key(self):
#         self.df = self.df.withColumn(self.surrogate_key_column, monotonically_increasing_id())
#         print("Surrogate key column added: " + self.surrogate_key_column)

#     def write_table(self):
#         self.df.write.format("delta") \
#             .mode("overwrite") \
#             .saveAsTable(self.target_table)
#         print("Feature table (with surrogate key) written: " + self.target_table)

#     def run(self):
#         self.read_source()
#         self.validate_key_uniqueness()
#         self.add_surrogate_key()
#         self.write_table()
#         return self.df


# class FeatureLabelSplitter:

#     def __init__(self, spark, source_table, primary_key, label_column,
#                  features_table, labels_table):
#         self.spark = spark
#         self.source_table = source_table
#         self.primary_key = primary_key
#         self.label_column = label_column
#         self.features_table = features_table
#         self.labels_table = labels_table
#         self.df = None

#     def read_source(self):
#         self.df = self.spark.table(self.source_table)

#     def split_and_write(self):
#         # X: everything except the label column
#         feature_columns = [c for c in self.df.columns if c != self.label_column]
#         features_df = self.df.select(*feature_columns)

#         # Y: primary key + label column only
#         labels_df = self.df.select(self.primary_key, self.label_column)

#         features_df.write.format("delta").mode("overwrite").saveAsTable(self.features_table)
#         labels_df.write.format("delta").mode("overwrite").saveAsTable(self.labels_table)

#         print("Features (X) table written: " + self.features_table)
#         print("Labels (Y) table written: " + self.labels_table)

#     def run(self):
#         self.read_source()
#         self.split_and_write()


# class FeatureStoreRegistrar:

#     def __init__(self, spark, features_table, feature_store_table, primary_key, description):
#         self.spark = spark
#         self.features_table = features_table
#         self.feature_store_table = feature_store_table
#         self.primary_key = primary_key
#         self.description = description
#         self.fe = FeatureEngineeringClient()

#     def register(self):
#         df = self.spark.table(self.features_table)

#         self.fe.create_table(
#             name=self.feature_store_table,
#             primary_keys=[self.primary_key],
#             df=df,
#             description=self.description
#         )

#         print("Feature store table created: " + self.feature_store_table)


# # ---------------------------
# # Notebook usage
# # ---------------------------

# # Step 1: Add surrogate key to feature_table
# preparer = FeatureTablePreparer(
#     spark=spark,
#     source_table="oil_gas_mlops.gold.feature_table",
#     target_table="oil_gas_mlops.gold.v1_feature_table",
#     key_columns=["transaction_date", "product_name", "destination_city"],
#     surrogate_key_column="feature_store_id"
# )
# preparer.run()

# # Step 2: Split into Features (X) and Labels (Y), both keyed by feature_store_id
# splitter = FeatureLabelSplitter(
#     spark=spark,
#     source_table="oil_gas_mlops.gold.v1_feature_table",
#     primary_key="feature_store_id",
#     label_column="total_demand_quantity",     # <-- CONFIRM this is your actual target column
#     features_table="oil_gas_mlops.gold.features_supply_chain",
#     labels_table="oil_gas_mlops.gold.labels_supply_chain"
# )
# splitter.run()

# # Step 3: Register ONLY the features table into the Feature Store
# registrar = FeatureStoreRegistrar(
#     spark=spark,
#     features_table="oil_gas_mlops.gold.features_supply_chain",
#     feature_store_table="oil_gas_mlops.gold.feature_store_supply_chain",
#     primary_key="feature_store_id",
#     description="Feature table (X only) - single source of truth for all ML consumers"
# )
# registrar.register()

In [0]:
from pyspark.sql.functions import col
from databricks.feature_engineering import FeatureEngineeringClient


# ============================================================
# 1. FEATURE TABLE PREPARER (cleans date type, validates the key)
# ============================================================

class FeatureTablePreparer:

    def __init__(self, spark, source_table, target_table, key_columns):
        self.spark = spark
        self.source_table = source_table
        self.target_table = target_table
        self.key_columns = key_columns
        self.df = None

    def read_source(self):
        self.df = self.spark.table(self.source_table)
        # Feature Store does not support timestamp_ntz, so cast to date
        self.df = self.df.withColumn("transaction_date", col("transaction_date").cast("date"))

    def validate_key_not_null(self):
        for key_col in self.key_columns:
            null_count = self.df.filter(col(key_col).isNull()).count()
            if null_count > 0:
                raise ValueError("Key column '" + key_col + "' has " + str(null_count) + " null values.")
        print("Key columns have no nulls: " + str(self.key_columns))

    def validate_key_uniqueness(self):
        total_rows = self.df.count()
        distinct_key_rows = self.df.select(*self.key_columns).distinct().count()

        if total_rows != distinct_key_rows:
            raise ValueError(
                "Key columns " + str(self.key_columns) + " are NOT unique. "
                "Total rows: " + str(total_rows) + ", distinct keys: " + str(distinct_key_rows)
            )
        print("Key uniqueness validated: " + str(total_rows) + " rows, all unique")

    def write_table(self):
        self.df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(self.target_table)
        print("Prepared table written: " + self.target_table)

    def run(self):
        self.read_source()
        self.validate_key_not_null()
        self.validate_key_uniqueness()
        self.write_table()


# ============================================================
# 2. FEATURE / LABEL SPLITTER (X table and Y table, both keyed)
# ============================================================

class FeatureLabelSplitter:

    def __init__(self, spark, source_table, key_columns, label_column,
                 features_table, labels_table):
        self.spark = spark
        self.source_table = source_table
        self.key_columns = key_columns
        self.label_column = label_column
        self.features_table = features_table
        self.labels_table = labels_table
        self.df = None

    def read_source(self):
        self.df = self.spark.table(self.source_table)

    def split_and_write(self):
        # X: every column except the label (key columns stay, they are part of the feature table)
        feature_columns = [c for c in self.df.columns if c != self.label_column]
        features_df = self.df.select(*feature_columns)

        # Y: key columns + label (the keys are needed for the lookup later)
        labels_df = self.df.select(*self.key_columns, self.label_column)

        features_df.write.format("delta").mode("overwrite") \
            .option("overwriteSchema", "true").saveAsTable(self.features_table)
        labels_df.write.format("delta").mode("overwrite") \
            .option("overwriteSchema", "true").saveAsTable(self.labels_table)

        print("Features (X) table written: " + self.features_table)
        print("Labels (Y) table written: " + self.labels_table)

    def run(self):
        self.read_source()
        self.split_and_write()


# ============================================================
# 3. FEATURE STORE REGISTRAR
# ============================================================

class FeatureStoreRegistrar:

    def __init__(self, spark, features_table, feature_store_table, key_columns, description):
        self.spark = spark
        self.features_table = features_table
        self.feature_store_table = feature_store_table
        self.key_columns = key_columns
        self.description = description
        self.fe = FeatureEngineeringClient()

    def register(self):
        df = self.spark.table(self.features_table)

        self.fe.create_table(
            name=self.feature_store_table,
            primary_keys=self.key_columns,
            df=df,
            description=self.description
        )
        print("Feature store table created: " + self.feature_store_table)


# ============================================================
# 4. RUN
# ============================================================

key_columns = ["transaction_date", "product_name", "destination_city"]
label_column = "total_demand_quantity"

# Step 1: Prepare and validate
preparer = FeatureTablePreparer(
    spark=spark,
    source_table="oil_gas_mlops.gold.feature_table",
    target_table="oil_gas_mlops.gold.v1_feature_table",
    key_columns=key_columns
)
preparer.run()

# Step 2: Split into Features (X) and Labels (Y), both carrying the key columns
splitter = FeatureLabelSplitter(
    spark=spark,
    source_table="oil_gas_mlops.gold.v1_feature_table",
    key_columns=key_columns,
    label_column=label_column,
    features_table="oil_gas_mlops.gold.features_supply_chain",
    labels_table="oil_gas_mlops.gold.labels_supply_chain"
)
splitter.run()

# Step 3: Register ONLY the features table in the Feature Store
registrar = FeatureStoreRegistrar(
    spark=spark,
    features_table="oil_gas_mlops.gold.features_supply_chain",
    feature_store_table="oil_gas_mlops.gold.feature_store_supply_chain",
    key_columns=key_columns,
    description="Feature table (X only) keyed by date, product and city"
)
registrar.register()